# 2.2 — Feature Engineering (Fixed)

**Perbaikan dari audit:**
- ✅ Lag hanya pada fitur yang benar-benar bulanan (bi_rate, inflasi)
- ✅ Autoregressive lags pada target (twp90_lag_1, _3, _6, _12)
- ✅ Fitur waktu: bulan_sin, bulan_cos, time_trend
- ✅ provinsi_id dipertahankan sebagai fitur (panel identity)
- ❌ TIDAK lag pada fitur tahunan (karena konstan, lag = nilai yang sama)

**Input:** `2_data_preprocessing/output/2.1_cleaned_data.csv`

**Output:** `2_data_preprocessing/output/2.2_final_feature_set.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '2_data_preprocessing' / 'output' / '2.1_cleaned_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find 2.1_cleaned_data.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '2_data_preprocessing' / 'output' / '2.1_cleaned_data.csv'
output_dir = ROOT / '2_data_preprocessing' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / '2.2_final_feature_set.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} cols')

In [ ]:
# === 1. Autoregressive lags pada TARGET (sangat penting untuk time-series) ===
ar_lags = [1, 2, 3, 6, 12]
for lag in ar_lags:
    df[f'twp90_lag_{lag}'] = df.groupby('provinsi_id')['twp90_pct'].shift(lag)
print(f'Created {len(ar_lags)} autoregressive lag features on twp90_pct')

# === 2. Lag pada fitur BULANAN saja (bi_rate dan inflasi) ===
monthly_features = ['x1_bi_rate_pct', 'x2_inflasi_yoy']
monthly_lags = [1, 3, 6]
for lag in monthly_lags:
    for feat in monthly_features:
        df[f'{feat}_lag_{lag}'] = df.groupby('provinsi_id')[feat].shift(lag)
print(f'Created {len(monthly_lags) * len(monthly_features)} monthly lag features')

# TIDAK membuat lag pada fitur tahunan (x3-x10) karena konstan per tahun
# shift(3) pada data konstan = nilai yang sama -> tidak informatif

# === 3. Fitur waktu (seasonality & trend) ===
df['bulan_sin'] = np.sin(2 * np.pi * df['bulan'] / 12)
df['bulan_cos'] = np.cos(2 * np.pi * df['bulan'] / 12)
df['time_trend'] = df.groupby('provinsi_id').cumcount()
print('Created time features: bulan_sin, bulan_cos, time_trend')

# === 4. Log-transform variabel dengan skala besar ===
df['log_pdrb'] = np.log1p(df['x3_pdrb_per_kapita'])
df['log_tabungan'] = np.log1p(df['x6_tabungan_miliar'])
print('Created log transforms: log_pdrb, log_tabungan')

# === 5. Isi residual missing (dari lag yang menghilangkan baris awal) ===
lag_cols = [c for c in df.columns if '_lag_' in c]
print(f'\n--- Missing Values on Lag Features ---')
display(df[lag_cols].isna().sum().sort_values(ascending=False).head(10))

# === 6. Simpan ===
assert (df['provinsi_id'] == 19).sum() == 0
df.to_csv(output_path, index=False)
print(f'\nSaved: {output_path}')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'All columns: {list(df.columns)}')